# Identificación causal — complementariedad JLoss × GaR sobre el spread

Cuatro estrategias sobre **una sola base**. La lectura normativa es **a favor de la necesidad de control macroprudencial**: la complementariedad observada es una externalidad que los bancos no internalizan; que el mercado la tarife *tarde y de forma supra-aditiva* en la cola severa es, precisamente, el argumento para regular ex-ante (no un elogio del laissez-faire). Requiere `causal_core.py` en esta carpeta y `linearmodels`.

In [ ]:
# === base principal all17 (5 países LatAm) ===
INFILE='Panel_final_all17.csv'
OUTDIR_NAME='causal_output_final17'

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, os
from pathlib import Path
import causal_core as cc
OUTDIR = Path(OUTDIR_NAME); (OUTDIR/'figuras').mkdir(parents=True, exist_ok=True)
def savefig(n): plt.savefig(OUTDIR/'figuras'/f'{n}.png', dpi=140, bbox_inches='tight')
P = cc.load(INFILE); CTR = cc.ctrls_in(P)
BOOT = 999   # repeticiones del wild bootstrap (baja a 199 para pruebas rápidas)
print('Base:', INFILE, '| países:', P.country.nunique(), '| controles:', CTR if CTR else 'ninguno')

## A. Inferencia con pocos clusters — *wild cluster bootstrap*

Cameron-Gelbach-Miller (2008): con pocos países, el *t* cluster/Driscoll-Kraay sobre-rechaza. El bootstrap restringido (pesos Rademacher por país, H0: θ=0) da el *p*-valor honesto de θ (interacción JLoss×GaR).

In [ ]:
specs = {'M2 (FE país+tiempo, sin controles)': []}
if CTR: specs['M3 (+ controles domésticos)'] = CTR
rowsA = []
for name, ct in specs.items():
    r = cc.wild_cluster_boot(P, ctrls=ct, B=BOOT)
    rowsA.append({'especificación':name,'θ':round(r['theta'],3),'SE cluster':round(r['se_cluster'],3),
                  't':round(r['t'],2),'p wild-boot':round(r['p_wildboot'],3),'clusters':r['G'],'n':r['n']})
tabA = pd.DataFrame(rowsA); tabA.to_csv(OUTDIR/'A_wildbootstrap.csv', index=False)
print('Lectura: comparar p-valor wild-boot con el p normal (~2*(1-Φ(|t|))).')
tabA

## B. Proyecciones locales estado-dependientes (Jordà; Ramey-Zubairy)

Respuesta impulso del EMBI a un *shock* de fragilidad (ΔJLoss) a horizonte h, distinta según el **régimen de cola** (severa vs. benigna, umbral en el percentil 30 del GaR), con FE país+tiempo, EMBI rezagado y SE cluster. El *timing* (shock en t, respuesta en t+h) atenúa la simultaneidad de las regresiones estáticas.

In [ ]:
lp, thr = cc.local_projections(P, H=8, sev_pct=0.30, ctrls=CTR)
lp.to_csv(OUTDIR/'B_local_projections.csv', index=False)
fig, ax = plt.subplots(figsize=(9,5.5))
for beta,se,lab,col in [('beta_sev','se_sev','Régimen cola severa (GaR bajo)','#c0392b'),
                        ('beta_ben','se_ben','Régimen benigno (GaR alto)','#27ae60')]:
    ax.plot(lp['h'], lp[beta], color=col, lw=2, marker='o', ms=4, label=lab)
    ax.fill_between(lp['h'], lp[beta]-1.645*lp[se], lp[beta]+1.645*lp[se], color=col, alpha=.15)
ax.axhline(0, color='grey', lw=.8)
ax.set_xlabel('horizonte h (trimestres tras el shock de JLoss)')
ax.set_ylabel('respuesta del EMBI (pb) a ΔJLoss')
ax.set_title('Proyecciones locales estado-dependientes: respuesta del spread a un shock de fragilidad\n'
             'bandas = IC90% (SE cluster por país)')
ax.legend(); savefig('B_irf_estado_dependiente'); plt.show()
print(f'Umbral de régimen severo: GaR_pp ≤ {thr:.2f}')
lp.round(3)

## C. Triple interacción con instituciones

θ_efectivo = `JxG_c` + `JxG_I`·(institución estandarizada). La hipótesis del *efecto idiosincrático local / necesidad de control* predice **amplificación mayor donde las instituciones son más débiles** (por tanto `JxG_I` > 0: más instituciones ⇒ θ menos negativo). ⚠️ `instituciones.csv` trae valores WGI **aproximados de plantilla**: reemplázalos por la vintage oficial antes de citar.

In [ ]:
rowsC = []
for var in ['rule_of_law','gov_effectiveness']:
    try:
        r = cc.triple_institucional(P, var=var, ctrls=CTR)
        jxg, jxg_se, jxg_t = r['JxG_c']; tri, tri_se, tri_t = r['JxG_I']
        th_low  = jxg + tri*(-1)   # instituciones débiles (z=-1)
        th_high = jxg + tri*(+1)   # instituciones fuertes (z=+1)
        rowsC.append({'institución':var,'θ (JxG_c)':round(jxg,3),'triple (JxG_I)':round(tri,3),
                      't triple':round(tri_t,2),'θ inst. débiles':round(th_low,3),
                      'θ inst. fuertes':round(th_high,3),'países':r['_meta']['paises']})
    except Exception as e:
        rowsC.append({'institución':var,'error':str(e)[:40]})
tabC = pd.DataFrame(rowsC); tabC.to_csv(OUTDIR/'C_triple_institucional.csv', index=False)
print('Si el triple (JxG_I) no es significativo, no hay evidencia de heterogeneidad institucional (poca potencia / datos plantilla).')
tabC

## D. IV shift-share con FE dobles

Se instrumenta la fragilidad con **exposición país × shock global** (auto-selección del shock con primera etapa más fuerte, típicamente `UST10Y_log`). Con FE de tiempo el nivel del shock global queda absorbido; identifica la variación **diferencial por exposición** (estilo Bartik). El objeto causal creíble es **β_JLoss** (nivel): el efecto de la interacción (θ_IV) queda mal instrumentado —segundo endógeno— y se reporta con cautela.

In [ ]:
r = cc.iv_shiftshare(P, global_var='auto', ctrls=CTR)
tabD = pd.DataFrame([{
  'shock global elegido': r.get('shock_global'),
  'F primera etapa': round(r.get('F_primera',float('nan')),1),
  'β_JLoss IV (nivel)': round(r.get('beta_JLoss_IV',float('nan')),3),
  'SE β_JLoss': round(r.get('se_JLoss',float('nan')),3),
  'p β_JLoss': round(r.get('p_JLoss',float('nan')),3),
  'θ_IV (interacción, cautela)': round(r.get('theta_IV',float('nan')),3),
  'n': r.get('n')}])
tabD.to_csv(OUTDIR/'D_iv_shiftshare.csv', index=False)
print('F>10 = instrumento fuerte. β_JLoss>0 y significativo = una subida exógena de fragilidad eleva el spread (dirección banco→soberano).')
tabD.T

## Síntesis

Lectura conjunta de las cuatro estrategias para esta base.

In [ ]:
print('Archivos generados en', OUTDIR.resolve())
for f in sorted(os.listdir(OUTDIR)):
    if f.endswith('.csv'): print('  ', f)
print('\nFiguras:'); [print('  ', f) for f in sorted(os.listdir(OUTDIR/'figuras'))]